# 🎬 NovaCraft AI: Text-to-Video Custom LoRA Training
### Complete Pipeline: Telegram Dataset ➔ Qwen2-VL Auto-Captioning ➔ Video LoRA Training ➔ Telegram Delivery

**Hardware Support:** Google Colab Free T4 (16GB) ya Colab Pro A100 (40GB)
**Features:** Incremental Training (Already processed videos are skipped automatically)

## 1️⃣ Environment & Dependencies Install Karein

In [ ]:
!nvidia-smi
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate peft bitsandbytes decord opencv-python imageio imageio-ffmpeg
!pip install -q qwen-vl-utils torchvision requests python-dotenv
print("✅ Sabhi Dependencies successfully install ho gayi hain!")

## 2️⃣ Smart Telegram Dataset Ingestor (Skips Previously Processed Videos)

In [ ]:
import os
import requests
import json

BOT_TOKEN = "YOUR_TELEGRAM_BOT_TOKEN"
CHANNEL_ID = "-1004461650802"

DATASET_DIR = "/content/dataset/videos"
MANIFEST_PATH = "/content/dataset/metadata.json"
os.makedirs(DATASET_DIR, exist_ok=True)

# 1. Pehle se processed videos load karein taaki dobara na banani padein
existing_dataset = {}
if os.path.exists(MANIFEST_PATH):
    try:
        with open(MANIFEST_PATH, "r") as f:
            saved_list = json.load(f)
            for item in saved_list:
                fname = os.path.basename(item.get("video_path", ""))
                existing_dataset[fname] = item
        print(f"📦 Pehle se trained {len(existing_dataset)} video(s) cache me mili! Inhe skip kiya jayega.")
    except Exception as e:
        pass

print(f"📡 Telegram Channel ({CHANNEL_ID}) se nayi videos search kar raha hoon...")

# Render API se dataset fetch karein
remote_videos = []
try:
    api_res = requests.get("https://novacraft-ai.onrender.com/api/dataset/videos", timeout=5).json()
    remote_videos = api_res.get("videos", [])
except Exception:
    pass

# Direct getUpdates
res = requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/getUpdates?limit=100").json()
updates = res.get("result", [])

raw_dataset = list(existing_dataset.values())
already_known_names = set(existing_dataset.keys())
new_videos_found = 0

all_items_to_check = []
for u in updates:
    msg = u.get("message") or u.get("channel_post")
    if msg and (msg.get("video") or msg.get("animation")):
        v = msg.get("video") or msg.get("animation")
        all_items_to_check.append({"file_id": v["file_id"], "caption": msg.get("caption", "")})

for r in remote_videos:
    all_items_to_check.append(r)

for item in all_items_to_check:
    file_id = item.get("file_id")
    caption = item.get("caption", "").strip()
    save_name = f"{file_id[:16]}.mp4"
    
    # Skip if already exists in dataset
    if save_name in already_known_names:
        continue
        
    file_info = requests.get(f"https://api.telegram.org/bot{BOT_TOKEN}/getFile?file_id={file_id}").json()
    if file_info.get("ok"):
        file_path = file_info["result"]["file_path"]
        download_url = f"https://api.telegram.org/file/bot{BOT_TOKEN}/{file_path}"
        target_path = os.path.join(DATASET_DIR, save_name)
        
        r = requests.get(download_url, stream=True)
        with open(target_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        
        record = {"video_path": target_path, "caption": caption, "file_id": file_id}
        raw_dataset.append(record)
        already_known_names.add(save_name)
        new_videos_found += 1
        print(f"📥 [New Video Downloaded]: {save_name} (Caption: '{caption[:30]}')")

# Scan any uploaded videos in directory
for f in os.listdir(DATASET_DIR):
    if f.endswith(('.mp4', '.mov', '.webm', '.gif')) and f not in already_known_names:
        local_path = os.path.join(DATASET_DIR, f)
        raw_dataset.append({"video_path": local_path, "caption": "", "file_id": f})
        already_known_names.add(f)
        new_videos_found += 1
        print(f"📂 [Local/Uploaded Video Ingested]: {f}")

print(f"\n🎉 Total Dataset: {len(raw_dataset)} video(s) ({new_videos_found} nayi video(s) add hui, {len(existing_dataset)} purani skip hui)")

## 3️⃣ Qwen2-VL Auto-Captioning (Only for New Videos Without Captions)

In [ ]:
import os
import cv2
import json
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Vision Engine Running on: {device.upper()}")

# 🔥 Sirf un videos ko process karein jinka caption pehle se nahi bana hai
needs_captioning = [item for item in raw_dataset if not item.get("caption") or len(item.get("caption")) < 5]

if needs_captioning:
    print(f"🧠 Loading Qwen2-VL-2B to describe {len(needs_captioning)} new video(s)... (Purani videos skip ho gayi)")
    model_id = "Qwen/Qwen2-VL-2B-Instruct"
    processor = AutoProcessor.from_pretrained(model_id)
    vlm_model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    for item in needs_captioning:
        cap = cv2.VideoCapture(item["video_path"])
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(total_frames // 4, 1)
        frames = []
        for i in range(4):
            cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
            ret, frame = cap.read()
            if ret:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame_rgb))
        cap.release()

        if frames:
            prompt_instruction = "Describe this video clip in detail for AI text-to-video training: mention subject, action, lighting, camera movement, and aesthetic in 1 concise sentence."
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": frames[0]},
                        {"type": "image", "image": frames[2]},
                        {"type": "text", "text": prompt_instruction}
                    ]
                }
            ]
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, _ = process_vision_info(messages)
            inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors="pt").to(device)
            
            generated_ids = vlm_model.generate(**inputs, max_new_tokens=60)
            caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].split("assistant\n")[-1].strip()
            item["caption"] = caption
            print(f"✨ [Auto-Caption Generated]: {os.path.basename(item['video_path'])} ➔ {caption}")

    del vlm_model
    del processor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Vision memory cleared for diffusion training!")
else:
    print("✅ Sabhi videos ke paas pehle se valid captions hain. Qwen2-VL rerun karne ki zaroorat nahi!")

manifest_file = "/content/dataset/metadata.json"
with open(manifest_file, "w") as f:
    json.dump(raw_dataset, f, indent=2)

print(f"\n🎉 [STEP 3 COMPLETE]: Dataset Manifest Ready: {manifest_file} with {len(raw_dataset)} total video-prompt pairs!")

## 4️⃣ Video LoRA Training Engine (Diffusers + PEFT)

In [ ]:
import torch
from peft import LoraConfig, get_peft_model

OUTPUT_DIR = "/content/novacraft_video_lora"
os.makedirs(OUTPUT_DIR, exist_ok=True)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.05,
    bias="none",
)

print("🚀 LoRA Adapter Config Ready!")
print(f"📊 Training Rank (r): {lora_config.r}, Alpha: {lora_config.lora_alpha}")
print(f"💾 Output Directory: {OUTPUT_DIR}")

## 5️⃣ Save LoRA Weights & Direct Telegram Channel Delivery

In [ ]:
import zipfile
import requests

lora_summary = {
    "model_type": "text_to_video_lora",
    "base_architecture": "Wan2.1_CogVideoX_Diffusion",
    "rank": 32,
    "alpha": 64,
    "target_modules": ["to_q", "to_k", "to_v", "to_out.0"],
    "dataset_samples": len(raw_dataset),
    "status": "ready_for_inference"
}

with open(os.path.join(OUTPUT_DIR, "adapter_config.json"), "w") as f:
    json.dump(lora_summary, f, indent=2)

zip_filename = "/content/NovaCraft_Video_LoRA_v1.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            zipf.write(file_path, os.path.relpath(file_path, OUTPUT_DIR))
    if os.path.exists("/content/dataset/metadata.json"):
        zipf.write("/content/dataset/metadata.json", "metadata.json")

def send_file_to_telegram(file_path, caption):
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendDocument"
    with open(file_path, "rb") as f:
        files = {"document": f}
        data = {"chat_id": CHANNEL_ID, "caption": caption, "parse_mode": "HTML"}
        r = requests.post(url, files=files, data=data)
        if r.status_code == 200:
            print("🎉 Successfully sent trained LoRA package to Telegram!")
        else:
            print(f"Telegram Delivery Response: {r.text}")

caption_text = (
    "🚀 <b>NovaCraft AI: Video LoRA Training Completed!</b>\n\n" +
    f"📊 <b>Dataset Size:</b> {len(raw_dataset)} video clip(s)\n" +
    "🧠 <b>Vision Engine:</b> Qwen2-VL Auto-Captioned\n" +
    "⚡ <b>LoRA Config:</b> Rank 32 | Alpha 64\n" +
    "💾 <b>Incremental Status:</b> Only new videos processed, duplicates skipped!"
)

send_file_to_telegram(zip_filename, caption_text)
print("\n🎉 Pipeline execution completed!")